In [ ]:
import sys
from pathlib import Path

# Allow imports from repository root (run from repo root or notebooks/)
_repo = Path.cwd()
if not (_repo / "run_simulation.py").exists():
    _repo = _repo.parent
if str(_repo) not in sys.path:
    sys.path.insert(0, str(_repo))



# Convergence plot

Cumulative **mean** of per-run **mean** total time (disaster → construction) vs number of Monte Carlo runs, with an approximate **95%** band on the mean (normal approx., 1.96 × SEM).

**Ways to get `run_mean_total`**

1. **Re-run here** — run the setup cell below (edit parameters to match [`run_simulation.ipynb`](run_simulation.ipynb) if needed).

2. **Load from disk** — after running the first cell of `run_simulation.ipynb`, save once:

   ```python
   np.savez_compressed("convergence_data.npz", run_mean_total=run_mean_total)
   ```

   Then set `DATA_NPZ = "convergence_data.npz"` in the setup cell and re-run only setup + plot (no full simulation).


In [ ]:
import importlib
import numpy as np

# --- Option A: load `run_mean_total` saved from `run_simulation.ipynb` (first cell), e.g.:
#     np.savez_compressed("convergence_data.npz", run_mean_total=run_mean_total)
DATA_NPZ = ""  # set to e.g. "convergence_data.npz" to skip re-running the simulation

# --- Option B: run Monte Carlo here (keep in sync with `run_simulation.ipynb` if you want identical settings)
NUM_PERMITS = 6571
N_RUNS = 200
BASE_SEED = 42
SIMULATION_DURATION = None
COLLECT_PERMITS = True

SCENARIO_PARAMS = {
    "name": "balanced_standard",
    "sequential": "standard",
    "ai_review": "none",
    "permit_mix": "la",
    "pre_application_distribution": "baseline",
    "review_duration_multipliers": None,
    "planning_staff_count": 20,
    "planning_caseload_per_staff": 7,
    "public_works_staff_count": 30,
    "public_works_caseload_per_staff": 7,
    "fire_staff_count": 10,
    "fire_caseload_per_staff": 7,
}

if DATA_NPZ:
    z = np.load(DATA_NPZ)
    run_mean_total = np.asarray(z["run_mean_total"], dtype=float)
    print(f"Loaded run_mean_total from {DATA_NPZ!r} (n={len(run_mean_total)})")
else:
    import run_simulation
    importlib.reload(run_simulation)
    from run_simulation import run_multiple_simulations

    print(
        f"Running {N_RUNS} simulations ({NUM_PERMITS} permits each, collect_permits={COLLECT_PERMITS})..."
    )
    results, _ = run_multiple_simulations(
        n_runs=N_RUNS,
        num_permits=NUM_PERMITS,
        simulation_duration=SIMULATION_DURATION,
        base_seed=BASE_SEED,
        scenario_params_list=[SCENARIO_PARAMS],
        collect_permits=COLLECT_PERMITS,
        collect_average_staff_utilization=True,
        utilization_kind="implied",
        utilization_step=0.05,
    )
    run_mean_total = np.array(
        [r["stats"]["average_total_time"]["mean"] for r in results],
        dtype=float,
    )
    print(f"Built run_mean_total from fresh runs (n={len(run_mean_total)})")


In [ ]:
# Convergence: cumulative mean of per-run mean total time vs number of runs (with 95% CI on the mean).
# Requires `run_mean_total` from the setup cell above.

import matplotlib.pyplot as plt
import pandas as pd

if "run_mean_total" not in globals():
    raise RuntimeError("Run the setup cell so `run_mean_total` exists.")

CHECKPOINTS = np.array([1, 5, 10, 25, 50, 75, 100, 150, 200], dtype=int)
n_avail = len(run_mean_total)
usable = CHECKPOINTS[CHECKPOINTS <= n_avail]
if len(usable) == 0:
    raise RuntimeError(f"Need at least one run; have n_avail={n_avail}.")

if n_avail < CHECKPOINTS.max():
    print(
        f"Note: only {n_avail} runs available; "
        f"increase N_RUNS to {CHECKPOINTS.max()} to plot every checkpoint."
    )

means = []
yerr = []
for k in usable:
    chunk = run_mean_total[:k]
    means.append(float(np.mean(chunk)))
    if k < 2:
        yerr.append(0.0)
    else:
        sem = float(np.std(chunk, ddof=1)) / np.sqrt(k)
        yerr.append(1.96 * sem)
means = np.array(means, dtype=float)
yerr = np.array(yerr, dtype=float)

# Build convergence table with summary statistics
ci_lower = means - yerr
ci_upper = means + yerr
std_devs = []
for k in usable:
    chunk = run_mean_total[:k]
    if k < 2:
        std_devs.append(0.0)
    else:
        std_devs.append(float(np.std(chunk, ddof=1)))

table = pd.DataFrame(
    {
        "Number of Runs": usable,
        "Mean (Days)": means,
        "Std. Dev.": std_devs,
        "95% CI Lower": ci_lower,
        "95% CI Upper": ci_upper,
    }
)

print("Convergence table:\n")
print(table.to_string(index=False, float_format="{:.2f}".format))

# Broken y-axis: small strip shows origin (not starting at data); top panel shows data clearly.
data_lo = float(np.nanmin(means - yerr))
data_hi = float(np.nanmax(means + yerr))
pad = max((data_hi - data_lo) * 0.08, 1.0)
ylim_top_lo = data_lo - pad
ylim_top_hi = data_hi + pad
strip_hi = min(100.0, max(5.0, ylim_top_lo * 0.06))
if strip_hi >= ylim_top_lo:
    strip_hi = max(ylim_top_lo / 2.0, 1.0)

color = "#4A7BA7"
fig, (ax_top, ax_bot) = plt.subplots(
    2,
    1,
    sharex=True,
    figsize=(10, 5.5),
    gridspec_kw={"height_ratios": [5, 1], "hspace": 0.06},
)

# Increase chart text sizes for readability.
title_fs = 18
label_fs = 14
tick_fs = 12

ax_top.errorbar(
    usable,
    means,
    yerr=yerr,
    fmt="-o",
    color=color,
    ecolor=color,
    capsize=4,
    markersize=6,
    linewidth=1.5,
)
ax_top.set_ylim(ylim_top_lo, ylim_top_hi)
ax_top.grid(True, alpha=0.35)
ax_top.set_title("Convergence of Mean Total Time vs Number of Simulation Runs", fontsize=title_fs)
ax_top.spines["bottom"].set_visible(False)
ax_top.tick_params(axis="x", which="both", bottom=False, labelbottom=False, labelsize=tick_fs)
ax_top.tick_params(axis="y", labelsize=tick_fs)
ax_top.set_ylabel("Mean total time to construction (days)", fontsize=label_fs)

ax_bot.set_ylim(0, strip_hi)
ax_bot.grid(True, alpha=0.35)
ax_bot.spines["top"].set_visible(False)
ax_bot.set_ylabel("")
ax_bot.set_xlabel("Number of simulation runs", fontsize=label_fs)
ax_bot.set_xlim(left=0)
ax_bot.tick_params(axis="both", labelsize=tick_fs)

d = 0.015
kwargs = dict(transform=ax_top.transAxes, color="0.35", clip_on=False, linewidth=1)
ax_top.plot((-d, +d), (-d, +d), **kwargs)
ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)
kwargs.update(transform=ax_bot.transAxes)
ax_bot.plot((-d, +d), (1 - d, 1 + d), **kwargs)
ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

fig.align_ylabels([ax_top, ax_bot])
plt.tight_layout()
plt.show()
